# 01 — Keşifsel Veri Analizi (EDA)

Bu defter projedeki üç ana veriyi inceler:

1. **Finansal seri**: XU100.IS kapanış fiyatı, log-return, gerçekleşen volatilite
2. **Haber akışı**: Kaynak (TCMB / FED / GDELT / mock) ve günlük dağılım
3. **NLP skorları**: Belirsizlik dağılımı, olay türü, belirsizlik-volatilite ilişkisi

Defteri çalıştırmadan önce:

```bash
python main.py --skip-fetch --force-mock --skip-viz
```

ile DB'nin dolu olduğundan emin ol.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

from src.pipeline.database import DatabaseManager

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

db = DatabaseManager(ROOT / config["database"]["path"])
ticker = config["financial"]["default_ticker"]
print(f"DB: {db}")
print(f"Ticker: {ticker}")

DB: DatabaseManager(path='/Users/omeraskin/cursorProject/data/thesis.db', financial=253, news=1578, scores=2630)
Ticker: XU100.IS


## 1. Finansal Seri

XU100.IS kapanış fiyatı, log-return ve 20 günlük gerçekleşen volatilite.

In [2]:
fin = db.get_financial_data(ticker)
fin["realized_vol_20"] = fin["log_return"].rolling(20).std()

print(f"Gözlem sayısı   : {len(fin)}")
print(f"Tarih aralığı   : {fin.index[0].date()} → {fin.index[-1].date()}")
print(f"Log return ort. : {fin['log_return'].mean():.4f}")
print(f"Log return std  : {fin['log_return'].std():.4f}")
print(f"Skewness        : {fin['log_return'].skew():.4f}")
print(f"Kurtosis (fazla): {fin['log_return'].kurtosis():.4f}")
fin.tail()

Gözlem sayısı   : 253
Tarih aralığı   : 2025-04-11 → 2026-04-14
Log return ort. : 0.1657
Log return std  : 1.4106
Skewness        : 0.5972
Kurtosis (fazla): 1.6164


,id,ticker,open,high,low,close,volume,log_return,created_at,realized_vol_20
date,,,,,,,,,,
2026-04-08,249,XU100.IS,13353.400391,13679.900391,13333.700195,13536.799805,9863100000,4.651159,2026-04-12 17:27:30,1.561448
2026-04-09,250,XU100.IS,13530.599609,13712.500000,13529.099609,13689.000000,8298400000,1.118070,2026-04-12 17:27:30,1.576877
2026-04-10,251,XU100.IS,13774.500000,14073.799805,13763.799805,14073.799805,10909800000,2.772231,2026-04-12 17:27:30,1.678229
2026-04-13,500,XU100.IS,13924.200195,14099.200195,13842.799805,14058.500000,10001000000,-0.108770,2026-04-15 21:47:29,1.630398
2026-04-14,501,XU100.IS,14170.000000,14356.200195,14098.400391,14202.200195,11794500000,1.016970,2026-04-15 21:47:29,1.602071


In [3]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(fin.index, fin["close"], color="#1a73e8", lw=1.2)
axes[0].set_ylabel("Kapanış (TL)")
axes[0].set_title(f"{ticker} — Kapanış Fiyatı")

colors = ["#e53935" if v < 0 else "#43a047" for v in fin["log_return"]]
axes[1].bar(fin.index, fin["log_return"], color=colors, width=1.0, alpha=0.7)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_ylabel("Log Return (%)")
axes[1].set_title("Günlük Log Return")

axes[2].plot(fin.index, fin["realized_vol_20"], color="#7b1fa2", lw=1.2)
axes[2].fill_between(fin.index, fin["realized_vol_20"], alpha=0.2, color="#7b1fa2")
axes[2].set_ylabel("Volatilite")
axes[2].set_title("20 Günlük Gerçekleşen Volatilite")

plt.tight_layout()
plt.show()

/var/folders/nl/fg3grmg96nsdnjts14zl46p80000gn/T/ipykernel_72064/3096364822.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fin["log_return"].dropna(), bins=40, color="#1a73e8",
             alpha=0.7, edgecolor="white")
axes[0].axvline(fin["log_return"].mean(), color="red", ls="--", label="Ort.")
axes[0].set_title("Log Return Dağılımı")
axes[0].set_xlabel("Log Return (%)")
axes[0].set_ylabel("Frekans")
axes[0].legend()

from scipy import stats
stats.probplot(fin["log_return"].dropna(), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (Normal Dağılım)")

plt.tight_layout()
plt.show()

/var/folders/nl/fg3grmg96nsdnjts14zl46p80000gn/T/ipykernel_72064/3912940812.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Haber Akışı

Kaynak dağılımı, günlük haber sayısı ve zaman serisi yoğunluğu.

In [5]:
news = db.get_news()
news["date"] = pd.to_datetime(news["date"])

print(f"Toplam haber: {len(news)}")
print(f"\nKaynak dağılımı:")
print(news["source"].value_counts())
print(f"\nGünlük haber ort: {len(news) / news['date'].nunique():.2f}")

Toplam haber: 1578

Kaynak dağılımı:
source
tcmb     639
yerel    471
fed      468
Name: count, dtype: int64

Günlük haber ort: 6.00


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

src_counts = news["source"].value_counts()
colors_src = ["#1a73e8", "#e53935", "#43a047", "#ff9800"][:len(src_counts)]
axes[0].bar(src_counts.index, src_counts.values, color=colors_src,
            edgecolor="white")
axes[0].set_title("Kaynak Dağılımı")
axes[0].set_ylabel("Haber Sayısı")
for i, v in enumerate(src_counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")

daily = news.groupby([news["date"].dt.date, "source"]).size().unstack(fill_value=0)
daily.plot(kind="area", stacked=True, ax=axes[1], alpha=0.7,
           color=colors_src[:len(daily.columns)])
axes[1].set_title("Günlük Haber Sayısı (Kaynak Bazında)")
axes[1].set_ylabel("Haber / gün")
axes[1].set_xlabel("Tarih")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

/var/folders/nl/fg3grmg96nsdnjts14zl46p80000gn/T/ipykernel_72064/1012563074.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. NLP Skorları

Belirsizlik skoru dağılımı, olay türü frekansı, impact_direction oranı.

In [7]:
scores = pd.read_sql_query(
    "SELECT date, event_type, uncertainty_score, impact_direction, model_used "
    "FROM nlp_scores",
    db.connection, parse_dates=["date"],
)

print(f"Toplam skor: {len(scores)}")
print(f"\nKullanılan modeller: {scores['model_used'].value_counts().to_dict()}")
print(f"\nBelirsizlik skoru özeti:")
print(scores["uncertainty_score"].describe().round(4))
print(f"\nImpact direction dağılımı: {scores['impact_direction'].value_counts().to_dict()}")

Toplam skor: 2630

Kullanılan modeller: {'mock': 1578, 'llama3': 1052}

Belirsizlik skoru özeti:
count    2630.0000
mean        0.4971
std         0.2702
min         0.0000
25%         0.2000
50%         0.5000
75%         0.7500
max         0.9462
Name: uncertainty_score, dtype: float64

Impact direction dağılımı: {-1: 1352, 0: 661, 1: 617}


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(scores["uncertainty_score"], bins=30, color="#ff9800",
             alpha=0.8, edgecolor="white")
axes[0].axvline(scores["uncertainty_score"].mean(), color="red", ls="--",
                label=f"Ort = {scores['uncertainty_score'].mean():.3f}")
axes[0].axvline(0.5, color="gray", ls=":", label="Nötr (0.5)")
axes[0].set_title("Belirsizlik Skoru Dağılımı")
axes[0].set_xlabel("Skor (0-1)")
axes[0].set_ylabel("Frekans")
axes[0].legend()

evt_counts = scores["event_type"].value_counts().head(10)
axes[1].barh(evt_counts.index[::-1], evt_counts.values[::-1],
             color="#7b1fa2", edgecolor="white")
axes[1].set_title("En Sık Olay Türleri (Top 10)")
axes[1].set_xlabel("Sayı")

imp_counts = scores["impact_direction"].value_counts().sort_index()
imp_labels = {-1: "Negatif", 0: "Nötr", 1: "Pozitif"}
colors_imp = {-1: "#e53935", 0: "#90a4ae", 1: "#43a047"}
axes[2].pie(imp_counts.values,
            labels=[imp_labels[i] for i in imp_counts.index],
            colors=[colors_imp[i] for i in imp_counts.index],
            autopct="%.1f%%", startangle=90, wedgeprops={"edgecolor": "white"})
axes[2].set_title("Impact Direction Dağılımı")

plt.tight_layout()
plt.show()

/var/folders/nl/fg3grmg96nsdnjts14zl46p80000gn/T/ipykernel_72064/2257126629.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Belirsizlik ↔ Volatilite İlişkisi

Günlük ortalama belirsizlik (`avg_uncertainty`) ile gerçekleşen volatilite arasındaki korelasyon.
Hipotez: **Yüksek belirsizlik → Yüksek volatilite**.

In [9]:
merged = db.get_merged_data(ticker)
merged["realized_vol_20"] = merged["log_return"].rolling(20).std()
merged["abs_return"] = merged["log_return"].abs()

valid = merged.dropna(subset=["avg_uncertainty", "realized_vol_20"])
print(f"Geçerli gözlem: {len(valid)}")

corr_vol = valid["avg_uncertainty"].corr(valid["realized_vol_20"])
corr_abs = valid["avg_uncertainty"].corr(valid["abs_return"])
corr_ret = valid["avg_uncertainty"].corr(valid["log_return"])

print(f"\nKorelasyonlar (Pearson):")
print(f"  Belirsizlik ↔ Realized Vol (20g): {corr_vol:+.4f}")
print(f"  Belirsizlik ↔ |Log Return|      : {corr_abs:+.4f}")
print(f"  Belirsizlik ↔ Log Return        : {corr_ret:+.4f}")

Geçerli gözlem: 234

Korelasyonlar (Pearson):
  Belirsizlik ↔ Realized Vol (20g): +0.0614
  Belirsizlik ↔ |Log Return|      : -0.0368
  Belirsizlik ↔ Log Return        : +0.0258


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(valid["avg_uncertainty"], valid["realized_vol_20"],
                alpha=0.5, c=valid["avg_uncertainty"], cmap="RdYlGn_r",
                edgecolors="white", s=40)
z = np.polyfit(valid["avg_uncertainty"], valid["realized_vol_20"], 1)
x_line = np.linspace(valid["avg_uncertainty"].min(),
                     valid["avg_uncertainty"].max(), 50)
axes[0].plot(x_line, np.poly1d(z)(x_line), "k--", lw=1.5, label="Trend")
axes[0].set_xlabel("Ortalama Belirsizlik")
axes[0].set_ylabel("Realized Vol (20g)")
axes[0].set_title(f"Belirsizlik ↔ Volatilite  (r = {corr_vol:+.3f})")
axes[0].legend()

ax2 = axes[1]
ax2b = ax2.twinx()
ax2.plot(valid.index, valid["realized_vol_20"], color="#7b1fa2",
         lw=1.5, label="Realized Vol")
ax2b.plot(valid.index, valid["avg_uncertainty"], color="#ff9800",
          lw=1.2, alpha=0.8, label="Belirsizlik")
ax2.set_ylabel("Realized Vol", color="#7b1fa2")
ax2b.set_ylabel("Belirsizlik", color="#ff9800")
ax2.set_title("Zaman Serisi — Volatilite ve Belirsizlik")
ax2.legend(loc="upper left")
ax2b.legend(loc="upper right")

plt.tight_layout()
plt.show()

/var/folders/nl/fg3grmg96nsdnjts14zl46p80000gn/T/ipykernel_72064/3187766765.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
bins = [0, 0.3, 0.5, 0.7, 1.01]
labels = ["Düşük (<0.3)", "Orta (0.3-0.5)", "Yüksek (0.5-0.7)", "Çok Yüksek (>0.7)"]
valid = valid.copy()
valid["unc_bin"] = pd.cut(valid["avg_uncertainty"], bins=bins, labels=labels)

group_stats = valid.groupby("unc_bin", observed=True).agg(
    n=("realized_vol_20", "size"),
    vol_mean=("realized_vol_20", "mean"),
    vol_std=("realized_vol_20", "std"),
    abs_ret_mean=("abs_return", "mean"),
).round(4)
group_stats

,n,vol_mean,vol_std,abs_ret_mean
unc_bin,,,,
Düşük (<0.3),16,1.1686,0.3719,0.8531
Orta (0.3-0.5),100,1.3696,0.3446,1.2256
Yüksek (0.5-0.7),104,1.3836,0.3605,1.0433
Çok Yüksek (>0.7),14,1.3330,0.4123,0.6222


## 5. Notlar

- Burada kullanılan belirsizlik skorları **MockNLP** tarafından üretilmiştir (keyword tabanlı).
- Gerçek Llama 3 skorlarıyla karşılaştırma için:
  1. `ollama serve` çalıştır, `ollama pull llama3`
  2. `python main.py --skip-fetch` (force-mock olmadan)
  3. Skorları DB'ye yazdıktan sonra bu defteri tekrar çalıştır
- Altın set değerlendirmesi: `python scripts/evaluate_nlp.py --gold data/gold_set.csv`

In [12]:
db.close()
print("EDA tamamlandı.")

EDA tamamlandı.
